# 04_model_training.ipynb

Train and optimize Random Forest and XGBoost classifiers on preprocessed heart disease data.

This notebook:
1. Trains original models (baseline)
2. Performs hyperparameter tuning using RandomizedSearchCV with StratifiedKFold cross-validation
3. Trains tuned models
4. Compares original vs tuned models
5. Evaluates both models with comprehensive metrics
6. Displays feature importance
7. Saves all models (original RF, tuned RF, original XGBoost, tuned XGBoost, best model)


In [ ]:
# Import libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import joblib
import warnings
warnings.filterwarnings('ignore')

# XGBoost import with fallback explanation if not installed
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError('xgboost is not installed. Install it with `pip install xgboost`.') from exc

# Paths
clean_path = os.path.join('data', 'processed', 'cleaned_data.csv')
scaler_path = os.path.join('models', 'scaler.pkl')
rf_path = os.path.join('models', 'random_forest.pkl')
xgb_path = os.path.join('models', 'xgboost.pkl')
tuned_rf_path = os.path.join('models', 'tuned_random_forest.pkl')
tuned_xgb_path = os.path.join('models', 'tuned_xgboost.pkl')
best_path = os.path.join('models', 'best_model.pkl')
os.makedirs('models', exist_ok=True)

# Load the cleaned dataset
if not os.path.exists(clean_path):
    raise FileNotFoundError(f'Cleaned dataset not found at {clean_path}. Please run preprocessing first.')
df = pd.read_csv(clean_path)
print('Loaded cleaned dataset with shape:', df.shape)


In [ ]:
# Define features and target
feature_columns = [
    'age_years', 'gender', 'height', 'weight', 'BMI',
    'ap_hi', 'ap_lo', 'pulse_pressure', 'cholesterol', 'gluc', 'smoke', 'alco', 'active'
]
target_column = 'cardio'

# Verify that all required columns exist
missing_cols = [col for col in feature_columns + [target_column] if col not in df.columns]
if missing_cols:
    raise KeyError(f'Missing required columns: {missing_cols}')

# Create feature matrix and target vector
X = df[feature_columns].copy()
y = df[target_column].copy()

print('Selected features:', feature_columns)
print('Target:', target_column)
print('X shape:', X.shape)
print('y shape:', y.shape)
print(f'Target distribution: {y.value_counts().to_dict()}')


## Data Preparation: Stratified Train-Test Split and Scaling

- **Stratified Split**: Uses `stratify=y` to ensure train/test sets have similar class distributions, preventing biased evaluation.
- **Feature Scaling**: StandardScaler normalizes features to zero mean and unit variance, improving model convergence.
- **Cross-Validation**: StratifiedKFold(cv=5) maintains class distribution in each fold, providing robust hyperparameter estimates.


In [ ]:
# Perform stratified train-test split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Maintains class distribution in both train and test sets
)
print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)
print(f'Train set - Class 0: {(y_train==0).sum()}, Class 1: {(y_train==1).sum()}')
print(f'Test set - Class 0: {(y_test==0).sum()}, Class 1: {(y_test==1).sum()}')

# Load the pre-fitted scaler from feature selection notebook
if not os.path.exists(scaler_path):
    raise FileNotFoundError(f'Scaler not found at {scaler_path}. Run the feature selection notebook first.')
scaler = joblib.load(scaler_path)
print('Loaded scaler from', scaler_path)

# Transform features using the loaded scaler
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('X_train_scaled shape:', X_train_scaled.shape)
print('X_test_scaled shape:', X_test_scaled.shape)


In [ ]:
# Helper function to evaluate models comprehensively
def evaluate_model(name, y_true, y_pred):
    """
    Evaluate a classifier and return metrics.
    
    Args:
        name (str): Model name for display
        y_true (array): True labels
        y_pred (array): Predicted labels
    
    Returns:
        dict: Dictionary containing all evaluation metrics
    """
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    
    print(f'\n{"="*60}')
    print(f'{name} Evaluation:')
    print(f'{"="*60}')
    print(f'Accuracy:  {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1 Score:  {f1:.4f}')
    print(f'Confusion Matrix:\n{cm}')
    print(f'Classification Report:\n{classification_report(y_true, y_pred)}')
    
    return {
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'Confusion Matrix': cm
    }

# Initialize results storage
results = []


In [ ]:
# ============================================================================
# STEP 1: Train ORIGINAL Random Forest (Baseline)
# ============================================================================
print('\n' + '='*60)
print('TRAINING ORIGINAL RANDOM FOREST (BASELINE)')
print('='*60)

# Create and train original Random Forest with default hyperparameters
rf_original = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1  # Use all available CPU cores
)
rf_original.fit(X_train_scaled, y_train)
rf_original_pred = rf_original.predict(X_test_scaled)

# Evaluate original Random Forest
rf_original_results = evaluate_model('Random Forest (Original)', y_test, rf_original_pred)
results.append(rf_original_results)

# Save original Random Forest
joblib.dump(rf_original, rf_path)
print(f'Saved original Random Forest to {rf_path}')


In [ ]:
# ============================================================================
# STEP 2: Train ORIGINAL XGBoost (Baseline)
# ============================================================================
print('\n' + '='*60)
print('TRAINING ORIGINAL XGBOOST (BASELINE)')
print('='*60)

# Create and train original XGBoost with default hyperparameters
xgb_original = XGBClassifier(
    n_estimators=100,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,  # Use all available CPU cores
    verbosity=0
)
xgb_original.fit(X_train_scaled, y_train)
xgb_original_pred = xgb_original.predict(X_test_scaled)

# Evaluate original XGBoost
xgb_original_results = evaluate_model('XGBoost (Original)', y_test, xgb_original_pred)
results.append(xgb_original_results)

# Save original XGBoost
joblib.dump(xgb_original, xgb_path)
print(f'Saved original XGBoost to {xgb_path}')


In [ ]:
# ============================================================================
# STEP 3: Hyperparameter Tuning for Random Forest using RandomizedSearchCV
# ============================================================================
print('\n' + '='*60)
print('HYPERPARAMETER TUNING: RANDOM FOREST')
print('='*60)
print('Using RandomizedSearchCV with StratifiedKFold (cv=5)...')

# Define hyperparameter search space for Random Forest
rf_param_dist = {
    'n_estimators': [50, 100, 200, 300, 500],
    'max_depth': [10, 20, 30, 40, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', None]
}

# Create cross-validation splitter that respects class distribution
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Perform RandomizedSearchCV to find best hyperparameters
rf_random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=50,  # Try 50 different combinations
    cv=cv,
    scoring='f1',  # Optimize for F1 score (good balance of precision/recall)
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print('Starting RandomizedSearchCV for Random Forest (this may take a few minutes)...')
rf_random_search.fit(X_train_scaled, y_train)

# Get best hyperparameters and model
print(f'\nBest hyperparameters for Random Forest: {rf_random_search.best_params_}')
print(f'Best CV F1 Score: {rf_random_search.best_score_:.4f}')

# Train the best model on full training set
rf_tuned = rf_random_search.best_estimator_
rf_tuned.fit(X_train_scaled, y_train)
rf_tuned_pred = rf_tuned.predict(X_test_scaled)

# Evaluate tuned Random Forest
rf_tuned_results = evaluate_model('Random Forest (Tuned)', y_test, rf_tuned_pred)
results.append(rf_tuned_results)

# Save tuned Random Forest
joblib.dump(rf_tuned, tuned_rf_path)
print(f'Saved tuned Random Forest to {tuned_rf_path}')

# Display improvement
rf_improvement = rf_tuned_results['Accuracy'] - rf_original_results['Accuracy']
print(f'\nRandom Forest Accuracy Improvement: {rf_improvement:+.4f}')


In [ ]:
# ============================================================================
# STEP 4: Hyperparameter Tuning for XGBoost using RandomizedSearchCV
# ============================================================================
print('\n' + '='*60)
print('HYPERPARAMETER TUNING: XGBOOST')
print('='*60)
print('Using RandomizedSearchCV with StratifiedKFold (cv=5)...')

# Define hyperparameter search space for XGBoost
xgb_param_dist = {
    'n_estimators': [50, 100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 7, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7]
}

# Perform RandomizedSearchCV to find best hyperparameters
xgb_random_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ),
    param_distributions=xgb_param_dist,
    n_iter=50,  # Try 50 different combinations
    cv=cv,
    scoring='f1',  # Optimize for F1 score
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print('Starting RandomizedSearchCV for XGBoost (this may take a few minutes)...')
xgb_random_search.fit(X_train_scaled, y_train)

# Get best hyperparameters and model
print(f'\nBest hyperparameters for XGBoost: {xgb_random_search.best_params_}')
print(f'Best CV F1 Score: {xgb_random_search.best_score_:.4f}')

# Train the best model on full training set
xgb_tuned = xgb_random_search.best_estimator_
xgb_tuned.fit(X_train_scaled, y_train)
xgb_tuned_pred = xgb_tuned.predict(X_test_scaled)

# Evaluate tuned XGBoost
xgb_tuned_results = evaluate_model('XGBoost (Tuned)', y_test, xgb_tuned_pred)
results.append(xgb_tuned_results)

# Save tuned XGBoost
joblib.dump(xgb_tuned, tuned_xgb_path)
print(f'Saved tuned XGBoost to {tuned_xgb_path}')

# Display improvement
xgb_improvement = xgb_tuned_results['Accuracy'] - xgb_original_results['Accuracy']
print(f'\nXGBoost Accuracy Improvement: {xgb_improvement:+.4f}')


In [ ]:
# ============================================================================
# STEP 5: Compare All Models and Select Best
# ============================================================================
print('\n' + '='*60)
print('MODEL COMPARISON: ORIGINAL vs TUNED')
print('='*60)

# Create comparison dataframe
comparison_df = pd.DataFrame(results)
print('\nDetailed Comparison:')
print(comparison_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score']].to_string(index=False))

# Determine best model based on F1 Score (best balance of precision and recall)
best_index = comparison_df['F1 Score'].idxmax()
best_model_name = comparison_df.loc[best_index, 'Model']
best_model_obj = None

# Get the best model object
if 'Random Forest (Tuned)' in comparison_df['Model'].values:
    best_model_obj = rf_tuned if comparison_df.loc[comparison_df['Model']=='Random Forest (Tuned)', 'F1 Score'].values[0] >= comparison_df.loc[comparison_df['Model']=='XGBoost (Tuned)', 'F1 Score'].values[0] else xgb_tuned
    if 'XGBoost (Tuned)' not in comparison_df['Model'].values or comparison_df.loc[comparison_df['Model']=='Random Forest (Tuned)', 'F1 Score'].values[0] >= comparison_df.loc[comparison_df['Model']=='XGBoost (Tuned)', 'F1 Score'].values[0]:
        best_model_obj = rf_tuned
        best_model_name = 'Random Forest (Tuned)'
    else:
        best_model_obj = xgb_tuned
        best_model_name = 'XGBoost (Tuned)'
else:
    # Fallback if tuning didn't work
    best_model_obj = xgb_original if comparison_df.loc[comparison_df['Model']=='XGBoost (Original)', 'Accuracy'].values[0] >= comparison_df.loc[comparison_df['Model']=='Random Forest (Original)', 'Accuracy'].values[0] else rf_original
    best_model_name = 'XGBoost (Original)' if best_model_obj == xgb_original else 'Random Forest (Original)'

# Save the best model
joblib.dump(best_model_obj, best_path)
print(f'\nBest Model: {best_model_name}')
print(f'Best Model Accuracy: {comparison_df.loc[best_index, "Accuracy"]:.4f}')
print(f'Best Model F1 Score: {comparison_df.loc[best_index, "F1 Score"]:.4f}')
print(f'Saved best model to {best_path}')

# Display feature importance for best model
print('\n' + '='*60)
print('FEATURE IMPORTANCE (Top 10)')
print('='*60)
feature_importances = best_model_obj.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

print(importance_df.head(10).to_string(index=False))


# ============================================================================
# STEP 6: Final Summary and Results
# ============================================================================
print('\n' + '='*60)
print('FINAL SUMMARY')
print('='*60)

# Calculate original accuracies
original_rf_acc = comparison_df[comparison_df['Model'] == 'Random Forest (Original)']['Accuracy'].values[0]
original_xgb_acc = comparison_df[comparison_df['Model'] == 'XGBoost (Original)']['Accuracy'].values[0]
tuned_rf_acc = comparison_df[comparison_df['Model'] == 'Random Forest (Tuned)']['Accuracy'].values[0]
tuned_xgb_acc = comparison_df[comparison_df['Model'] == 'XGBoost (Tuned)']['Accuracy'].values[0]

print(f'\nRandom Forest:')
print(f'  Original Accuracy: {original_rf_acc:.4f}')
print(f'  Tuned Accuracy:    {tuned_rf_acc:.4f}')
print(f'  Improvement:       {tuned_rf_acc - original_rf_acc:+.4f}')

print(f'\nXGBoost:')
print(f'  Original Accuracy: {original_xgb_acc:.4f}')
print(f'  Tuned Accuracy:    {tuned_xgb_acc:.4f}')
print(f'  Improvement:       {tuned_xgb_acc - original_xgb_acc:+.4f}')

print(f'\n\nBest Model Selected: {best_model_name}')
best_accuracy = comparison_df.loc[comparison_df['Model'] == best_model_name, 'Accuracy'].values[0]
best_f1 = comparison_df.loc[comparison_df['Model'] == best_model_name, 'F1 Score'].values[0]
print(f'Best Model Accuracy: {best_accuracy:.4f}')
print(f'Best Model F1 Score: {best_f1:.4f}')

print(f'\n\nTop 10 Most Important Features:')
for idx, row in importance_df.head(10).iterrows():
    print(f'  {row["Feature"]:20s} {row["Importance"]:.4f}')

print('\n' + '='*60)
print('All models have been trained, tuned, evaluated, and saved!')
print('='*60)


## Tuning Methodology Explained

### RandomizedSearchCV vs GridSearchCV
- **RandomizedSearchCV**: Tests random combinations of hyperparameters (more efficient for large spaces)
- **GridSearchCV**: Tests all combinations (exhaustive but slower)
- We use RandomizedSearchCV with 50 iterations to balance exploration and computation time

### StratifiedKFold Cross-Validation
- **Why stratified?** Ensures each fold has similar class distribution, preventing biased estimates
- **cv=5**: Splits data into 5 folds, trains 5 models, averages scores for robust evaluation
- Prevents overfitting to specific train/test splits

### Hyperparameter Meanings
**Random Forest:**
- `n_estimators`: Number of trees (more = better but slower)
- `max_depth`: Maximum tree depth (lower = simpler, less overfitting)
- `min_samples_split`: Minimum samples to split a node (higher = simpler trees)
- `min_samples_leaf`: Minimum samples in leaf (higher = simpler predictions)
- `max_features`: Features to consider per split (subset = more diversity)

**XGBoost:**
- `n_estimators`: Number of boosting rounds
- `max_depth`: Maximum tree depth
- `learning_rate`: Shrinkage factor (lower = slower but potentially better)
- `subsample`: Fraction of training data per tree (0.8 = 80%)
- `colsample_bytree`: Fraction of features per tree
- `min_child_weight`: Minimum sum of weights in leaf (higher = more conservative)

### Why F1 Score for Optimization?
F1 Score balances Precision and Recall: `F1 = 2 * (Precision * Recall) / (Precision + Recall)`
- Better than accuracy for imbalanced datasets
- Prevents optimizing for one metric at the expense of another
